In [36]:
import os

PROJECT_DIR = (
    "/work/archcomplab/ssaraf/"
    "Maya Archarological Sites/Model Training/"
    "swin3d_project"
)

os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())

Working directory: /work/archcomplab/ssaraf/Maya Archarological Sites/Model Training/swin3d_project


In [37]:
from pathlib import Path

import torch

from models.swin3d_model import Model
from datasets.belize_laz_dataset import create_dataloaders
from evaluate import evaluate_best_model

In [38]:
train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=8,
    voxel_max=65_000,
    num_workers=4,
)

print("Test batches:", len(test_loader))

Test batches: 10


In [39]:
train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=2,
    voxel_max=65_000,
    num_workers=4,
)

In [40]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = Model(
    num_classes=3,
    in_channels=4,
).to(device)

print("Device:", device)

Device: cuda


In [42]:
BEST_CHECKPOINT = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_checkpoints/"
    "dice_soft_avd/"
    "swin3d_dice_soft_avd_best_04.pth"
)

RESULTS_DIR = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_results/dice_soft_avd"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print("Checkpoint:", BEST_CHECKPOINT)
print("Results directory:", RESULTS_DIR)

Checkpoint exists: True
Checkpoint: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/dice_soft_avd/swin3d_dice_soft_avd_best_04.pth
Results directory: /mnt/nrdstor/archcomplab/ssaraf/swin3d_results/dice_soft_avd


In [43]:
(
    metrics,
    results_df,
    per_class_df,
    confusion_matrix_df,
    saved_paths,
) = evaluate_best_model(
    model=model,
    test_loader=test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=RESULTS_DIR,
    device=device,
    class_names=[
        "Ground",
        "Structure",
        "Plazuela",
    ],
    print_progress=True,
    save_results=True,
)

Loaded best checkpoint from:
/mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/dice_soft_avd/swin3d_dice_soft_avd_best_04.pth
Checkpoint epoch: 61
Evaluated test batch 1/40
Evaluated test batch 5/40
Evaluated test batch 10/40
Evaluated test batch 15/40
Evaluated test batch 20/40
Evaluated test batch 25/40
Evaluated test batch 30/40
Evaluated test batch 35/40
Evaluated test batch 40/40

BEST-CHECKPOINT TEST RESULTS
Best epoch:        61
Overall Accuracy:  0.928498
F1 Average:        0.549837
Precision Average: 0.538774
Recall Average:    0.601590
mIoU:              0.458230

Ground
  Precision: 0.984113
  Recall:    0.945061
  F1:        0.964192
  IoU:       0.930859
  Support:   4,897,010
  Confusion: TP=4,627,974, FP=74,713, FN=269,036, TN=228,277

Structure
  Precision: 0.221715
  Recall:    0.122780
  F1:        0.158041
  IoU:       0.085800
  Support:   37,555
  Confusion: TP=4,611, FP=16,186, FN=32,944, TN=5,146,259

Plazuela
  Precision: 0.410494
  Recall:    0.736930
  F1:   

In [44]:
display(results_df)

,Test Set: Overall (Pixel / Voxel) Accuracy (OA),Test Set: F1 Average,Test Set: F1 - Ground,Test Set: F1 - Structure,Test Set: F1 - Plazuelas,Test Set: Precision Average,Test Set: Precision - Ground,Test Set: Precision - Structures,Test Set: Precision - Plazuelas,Test Set: Recall Average,Test Set: Recall - Ground,Test Set: Recall - Structure,Test Set: Recall - Plazuelas,Test Set: mIoU (mean Intersection over Union),Test Set: IoU Ground,Test Set: IoU Structure,Test Set: IoU Plazuela,Test Set: Confusion Matrix TP / FP / FN / TN: Ground,Test Set: Confusion Matrix TP / FP / FN / TN: Structures,Test Set: Confusion Matrix TP / FP / FN / TN: Plazuelas
0,0.928498,0.549837,0.964192,0.158041,0.527277,0.538774,0.984113,0.221715,0.410494,0.60159,0.945061,0.12278,0.73693,0.45823,0.930859,0.0858,0.358029,TP=4627974; FP=74713; FN=269036; TN=228277,TP=4611; FP=16186; FN=32944; TN=5146259,TP=195607; FP=280909; FN=69828; TN=4653656


In [45]:
display(per_class_df)

,Class,Precision,Recall,F1,IoU,TP,FP,FN,TN,Support
0,Ground,0.984113,0.945061,0.964192,0.930859,4627974,74713,269036,228277,4897010
1,Structure,0.221715,0.122780,0.158041,0.085800,4611,16186,32944,5146259,37555
2,Plazuela,0.410494,0.736930,0.527277,0.358029,195607,280909,69828,4653656,265435


In [46]:
display(confusion_matrix_df)

,Predicted Ground,Predicted Structure,Predicted Plazuela
Actual Ground,4627974,5650,263386
Actual Structure,15421,4611,17523
Actual Plazuela,59292,10536,195607


In [47]:
print(
    "Overall accuracy:",
    metrics["overall_accuracy"]
)

print(
    "Structure F1:",
    metrics["per_class"]["Structure"]["f1"]
)

print(
    "Plazuela IoU:",
    metrics["per_class"]["Plazuela"]["iou"]
)

Overall accuracy: 0.9284984615384615
Structure F1: 0.1580408554976693
Plazuela IoU: 0.35802900736532295
